In [ ]:
import re
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import joblib

In [3]:
LOG_PATTERN = re.compile(
    r'(?P<ip>\S+) - - \[(?P<timestamp>.*?)\] '
    r'"(?P<method>\S+) (?P<path>\S+) (?P<protocol>\S+)" '
    r'(?P<status>\d+) (?P<size>\d+) '
    r'"(?P<referer>.*?)" "(?P<agent>.*?)" (?P<resp_time>\d+)'
)

In [4]:
def parse_log_file(filepath):
    logs = []

    with open(filepath, "r") as f:
        for line in tqdm(f):
            match = LOG_PATTERN.match(line)

            if match:
                data = match.groupdict()

                logs.append({
                    "ip": data["ip"],
                    "method": data["method"],
                    "path": data["path"],
                    "status": int(data["status"]),
                    "size": int(data["size"]),
                    "resp_time": int(data["resp_time"]),
                    "agent": data["agent"]
                })

    return pd.DataFrame(logs)

In [5]:
df = parse_log_file("../resources/logfiles.log")

df.head()

1000000it [00:03, 274401.80it/s]


,ip,method,path,status,size,resp_time,agent
0,233.223.117.90,DELETE,/usr/admin,502,4963,45,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...
1,162.253.4.179,GET,/usr/admin/developer,200,5041,3885,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...
2,252.156.232.172,POST,/usr/register,404,5028,3350,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...
3,182.215.249.159,PUT,/usr/register,304,4936,767,Mozilla/5.0 (Android 10; Mobile; rv:84.0) Geck...
4,160.36.208.51,POST,/usr,304,4979,84,Mozilla/5.0 (Linux; Android 10; ONEPLUS A6000)...


In [6]:
METHOD_MAP = {
    "GET":0,
    "POST":1,
    "PUT":2,
    "DELETE":3,
    "PATCH":4
}

def create_features(df):
    features = pd.DataFrame()
    features["method"] = df["method"].map(METHOD_MAP).fillna(-1)
    features["path_length"] = df["path"].apply(len)
    features["status"] = df["status"]
    features["response_size"] = df["size"]
    features["response_time"] = df["resp_time"]
    features["has_admin"] = df["path"].str.contains("admin").astype(int)
    features["has_login"] = df["path"].str.contains("login").astype(int)
    features["has_config"] = df["path"].str.contains("config").astype(int)
    features["is_error"] = (df["status"] >= 400).astype(int)

    return features

In [7]:
X = create_features(df)

X.head()

,method,path_length,status,response_size,response_time,has_admin,has_login,has_config,is_error
0,3,10,502,4963,45,1,0,0,1
1,0,20,200,5041,3885,1,0,0,0
2,1,13,404,5028,3350,0,0,0,1
3,2,13,304,4936,767,0,0,0,0
4,1,4,304,4979,84,0,0,0,0


In [8]:
X[X['method'] == -1]

,method,path_length,status,response_size,response_time,has_admin,has_login,has_config,is_error


In [9]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [10]:
model = IsolationForest(
    n_estimators=100,
    contamination=0.05,
    random_state=42
)
model.fit(X_scaled)

,n_estimators,100
,max_samples,'auto'
,contamination,0.05
,max_features,1.0
,bootstrap,False
,n_jobs,None
,random_state,42
,verbose,0
,warm_start,False


In [11]:
preds = model.predict(X_scaled)

df["anomaly"] = preds

In [12]:
scores = model.decision_function(X_scaled)

df["anomaly_score"] = scores

In [13]:
df[df["anomaly"] == -1].head(20)

,ip,method,path,status,size,resp_time,agent,anomaly,anomaly_score
15,154.131.45.155,GET,/usr/login,200,5059,607,Mozilla/5.0 (Android 10; Mobile; rv:84.0) Geck...,-1,-0.028970
26,71.158.198.139,DELETE,/usr/login,304,5077,516,Mozilla/5.0 (Android 10; Mobile; rv:84.0) Geck...,-1,-0.014440
31,102.247.49.87,GET,/usr/login,200,4959,2248,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,-1,-0.002495
65,93.234.0.101,POST,/usr/login,200,5088,296,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_9_3)...,-1,-0.037848
144,5.80.243.143,POST,/usr/login,304,5144,3475,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:8...,-1,-0.016746
158,131.1.151.255,GET,/usr/login,200,5080,3957,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,-1,-0.028363
161,153.56.100.21,DELETE,/usr/login,200,5059,330,Mozilla/5.0 (Linux; Android 10; ONEPLUS A6000)...,-1,-0.046992
179,21.176.104.215,POST,/usr,500,5156,989,Mozilla/5.0 (Android 10; Mobile; rv:84.0) Geck...,-1,-0.008946
189,29.171.152.208,POST,/usr/login,200,4888,3961,Mozilla/5.0 (Linux; Android 10; ONEPLUS A6000)...,-1,-0.033117
249,57.178.191.99,GET,/usr/login,200,5000,425,Mozilla/5.0 (iPhone; CPU iPhone OS 12_4_9 like...,-1,-0.028111


In [ ]:
joblib.dump(model, "anomaly_model.pkl")